# RMSX Molstar Widget Sizing Prototype

This notebook tests a notebook-display path that follows the pattern used by mature notebook viewers: give Jupyter a real widget with explicit dimensions, then render the Molstar FlipBook inside that stable output box.

Use this to compare against the raw iframe display if the viewer appears low, cropped, or off-center in JupyterLab.

## 1. Load the notebook helper

Restart the kernel before testing this after code changes so Jupyter does not reuse an old embedded viewer.

In [ ]:
from pathlib import Path
import importlib
import math
import sys

PROJECT = Path.cwd()
if not (PROJECT / "tools" / "rmsx").exists():
    PROJECT = Path("/Users/finn/Documents/Flipbook Integration")

sys.path.insert(0, str(PROJECT / "tools" / "rmsx"))

import rmsx_molstar_notebook
importlib.reload(rmsx_molstar_notebook)
from rmsx_molstar_notebook import view_flipbook_snapshots

OUTPUT = PROJECT / "notebook_outputs"
OUTPUT.mkdir(exist_ok=True)
PROJECT

## 2. Choose or create snapshot PDBs

If the earlier RMSX notebook output exists, this uses those real RMSX slice PDBs. Otherwise it creates five prototype snapshots from the bundled 1UBQ PDB by varying B-factors, which is enough to test Molstar sizing, centering, palettes, and controls.

In [ ]:
def write_1ubq_bfactor_snapshot(source_pdb: Path, target_pdb: Path, phase: float) -> None:
    lines = []
    for line in source_pdb.read_text(encoding="utf-8").splitlines():
        if line.startswith(("ATOM  ", "HETATM")):
            try:
                residue_id = int(line[22:26])
            except ValueError:
                residue_id = 0
            bfactor = 0.25 + 4.5 * ((math.sin(residue_id / 7.0 + phase) + 1.0) / 2.0)
            lines.append(f"{line[:60]}{bfactor:6.2f}{line[66:]}")
        else:
            lines.append(line)
    target_pdb.write_text("\n".join(lines) + "\n", encoding="utf-8")


real_snapshot_dir = OUTPUT / "rmsx_1ubq" / "chain_7_rmsx"
generated_snapshot_dir = OUTPUT / "widget_sizing_snapshots"

if real_snapshot_dir.exists() and list(real_snapshot_dir.glob("*.pdb")):
    snapshot_dir = real_snapshot_dir
else:
    source = PROJECT / "tools" / "rmsx" / "test-data" / "1UBQ.pdb"
    generated_snapshot_dir.mkdir(parents=True, exist_ok=True)
    for index, phase in enumerate([0.0, 0.9, 1.8, 2.7, 3.6], start=1):
        write_1ubq_bfactor_snapshot(source, generated_snapshot_dir / f"slice_{index}_first_frame.pdb", phase)
    snapshot_dir = generated_snapshot_dir

snapshot_files = sorted(snapshot_dir.glob("*.pdb"))
print(f"Using {len(snapshot_files)} snapshots from {snapshot_dir}")
snapshot_files[:3]

## 3. Build the FlipBook result

This uses the same `rmsx-molstar-viewer/v1` manifest and the same vendored Molstar assets as the Galaxy/native viewer. It also writes a standalone HTML file so you can compare notebook rendering against a normal browser tab.

In [ ]:
result = view_flipbook_snapshots(
    snapshot_dir,
    palette="turbo",
    title="RMSX Molstar Widget Sizing Prototype",
    height=920,
    write_html=OUTPUT / "rmsx_widget_sizing_prototype.html",
)

print(result.manifest["schemaVersion"])
print(f"{len(result.manifest['slices'])} slices")
print(result.html_path)

## 4. Direct no-iframe display

Test this cell first. It mounts Molstar directly into a normal notebook output `<div>` instead of using an iframe. This is closest to the py3Dmol-style approach and should avoid the JupyterLab iframe sizing path that was pushing the structures to the bottom. Use the **Fit** button if JupyterLab finishes laying out the output after Molstar initializes.

In [ ]:
result.display_direct(height=920, spacing=0.7, thickness=1.0)

## 5. Widget-wrapped iframe comparison

This keeps the previous widget wrapper. If this still looks bottom-aligned but the direct display above is centered, then the iframe path is the problem.

In [ ]:
result.display_widget(height=920)

## 6. Raw iframe comparison

Run this cell only when you want to compare the original iframe-only path. This is expected to behave like the earlier notebook output in JupyterLab.

In [ ]:
result.display()

## 7. Browser fallback check

If both notebook outputs look wrong, open the standalone HTML file. If the browser file is centered, the remaining problem is notebook/JupyterLab embedding. If the browser file is also off-center, the viewer reset logic needs another pass.

In [ ]:
from IPython.display import FileLink

FileLink(result.html_path)